In [1]:
import os
import pandas as pd
import psycopg2
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

os.chdir(r'C:\Users\bamla\OneDrive\Desktop\fintech-review-analytics')
print("Libraries loaded!")

Libraries loaded!


In [2]:
df = pd.read_csv('data/raw/reviews_with_sentiment.csv')
print(f"Loaded {len(df):,} reviews")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 1,797 reviews
Columns: ['review_id', 'review', 'rating', 'date', 'bank', 'source', 'sentiment_score', 'sentiment_label', 'identified_theme']


,review_id,review,rating,date,bank,source,sentiment_score,sentiment_label,identified_theme
0,1,pels,5,2026-05-18,CBE,Google Play,0.0000,Neutral,General Feedback
1,2,What an excellent app with smooth performance !!,5,2026-05-18,CBE,Google Play,0.6467,Positive,UI & App Design
2,3,በጣም ጥሩ ነው እነማሰግነለን,5,2026-05-18,CBE,Google Play,0.0000,Neutral,General Feedback
3,4,svabst keessatti argamu yoo ta'u yeroo ammaa k...,5,2026-05-18,CBE,Google Play,0.0000,Neutral,General Feedback
4,5,nays,5,2026-05-17,CBE,Google Play,0.0000,Neutral,General Feedback


## Task 3: PostgreSQL Database Engineering

### Schema Design
- **banks** table: metadata about the three banks
- **reviews** table: all scraped and processed review data

### Relationships
- reviews.bank_id → banks.bank_id (FOREIGN KEY)

In [3]:
# Database connection settings
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'database': 'bank_reviews',
    'user':     'postgres',
    'password': 'Ma192837465#'  # ← replace with your password
}

# Test connection
try:
    conn = psycopg2.connect(**DB_CONFIG)
    print("✅ Connected to PostgreSQL successfully!")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")

✅ Connected to PostgreSQL successfully!


In [4]:
# Create tables
schema_sql = """
-- Drop tables if they exist (clean slate)
DROP TABLE IF EXISTS reviews CASCADE;
DROP TABLE IF EXISTS banks CASCADE;

-- Banks table
CREATE TABLE banks (
    bank_id     SERIAL PRIMARY KEY,
    bank_name   VARCHAR(100) NOT NULL UNIQUE,
    app_name    VARCHAR(200),
    app_id      VARCHAR(200),
    avg_rating  DECIMAL(3,2),
    created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Reviews table
CREATE TABLE reviews (
    review_id        INTEGER PRIMARY KEY,
    bank_id          INTEGER REFERENCES banks(bank_id),
    review_text      TEXT,
    rating           INTEGER CHECK (rating BETWEEN 1 AND 5),
    review_date      DATE,
    sentiment_label  VARCHAR(20),
    sentiment_score  DECIMAL(6,4),
    identified_theme VARCHAR(100),
    source           VARCHAR(50) DEFAULT 'Google Play',
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

try:
    cur.execute(schema_sql)
    conn.commit()
    print("✅ Schema created successfully!")
    print("   - Table 'banks' created")
    print("   - Table 'reviews' created")
except Exception as e:
    conn.rollback()
    print(f"❌ Schema creation failed: {e}")
finally:
    cur.close()
    conn.close()

✅ Schema created successfully!
   - Table 'banks' created
   - Table 'reviews' created


In [5]:
# Bank metadata
banks_data = [
    ('CBE',    'Commercial Bank of Ethiopia Mobile', 'com.combanketh.mobilebanking'),
    ('BOA',    'Bank of Abyssinia Mobile Banking',   'com.boa.boaMobileBanking'),
    ('Dashen', 'Dashen Bank Super App',              'com.dashen.dashensuperapp'),
]

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

try:
    for bank_name, app_name, app_id in banks_data:
        avg_rating = float(df[df['bank'] == bank_name]['rating'].mean())  # ← fixed
        cur.execute("""
            INSERT INTO banks (bank_name, app_name, app_id, avg_rating)
            VALUES (%s, %s, %s, %s)
            ON CONFLICT (bank_name) DO NOTHING
        """, (bank_name, app_name, app_id, round(avg_rating, 2)))
    
    conn.commit()
    print("✅ Banks table populated!")
    
    # Verify
    cur.execute("SELECT * FROM banks")
    rows = cur.fetchall()
    print("\nBanks table contents:")
    for row in rows:
        print(f"  {row}")
        
except Exception as e:
    conn.rollback()
    print(f"❌ Insert failed: {e}")
finally:
    cur.close()
    conn.close()

✅ Banks table populated!

Banks table contents:
  (1, 'CBE', 'Commercial Bank of Ethiopia Mobile', 'com.combanketh.mobilebanking', Decimal('4.09'), datetime.datetime(2026, 5, 19, 10, 43, 6, 518101))
  (2, 'BOA', 'Bank of Abyssinia Mobile Banking', 'com.boa.boaMobileBanking', Decimal('3.52'), datetime.datetime(2026, 5, 19, 10, 43, 6, 518101))
  (3, 'Dashen', 'Dashen Bank Super App', 'com.dashen.dashensuperapp', Decimal('3.93'), datetime.datetime(2026, 5, 19, 10, 43, 6, 518101))


In [6]:
# Get bank_id mapping
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

try:
    cur.execute("SELECT bank_id, bank_name FROM banks")
    bank_map = {row[1]: row[0] for row in cur.fetchall()}
    print("Bank ID mapping:", bank_map)
    
    # Insert reviews in batches
    inserted = 0
    errors = 0
    
    for _, row in df.iterrows():
        try:
            bank_id = bank_map.get(row['bank'])
            cur.execute("""
                INSERT INTO reviews (
                    review_id, bank_id, review_text, rating,
                    review_date, sentiment_label, sentiment_score,
                    identified_theme, source
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (review_id) DO NOTHING
            """, (
                int(row['review_id']),
                bank_id,
                str(row['review']),
                int(row['rating']),
                str(row['date']),
                str(row['sentiment_label']),
                float(row['sentiment_score']),
                str(row['identified_theme']),
                str(row['source'])
            ))
            inserted += 1
        except Exception as e:
            errors += 1
    
    conn.commit()
    print(f"\n✅ Reviews inserted: {inserted:,}")
    print(f"   Errors: {errors}")
    
except Exception as e:
    conn.rollback()
    print(f"❌ Insert failed: {e}")
finally:
    cur.close()
    conn.close()

Bank ID mapping: {'CBE': 1, 'BOA': 2, 'Dashen': 3}

✅ Reviews inserted: 1,797
   Errors: 0


In [7]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print("=== Database Verification ===\n")

# 1. Count reviews per bank
cur.execute("""
    SELECT b.bank_name, COUNT(r.review_id) as review_count
    FROM banks b
    LEFT JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY review_count DESC
""")
print("Reviews per bank:")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]:,}")

# 2. Average rating per bank
cur.execute("""
    SELECT b.bank_name, 
           ROUND(AVG(r.rating)::numeric, 2) as avg_rating,
           ROUND(AVG(r.sentiment_score)::numeric, 3) as avg_sentiment
    FROM banks b
    JOIN reviews r ON b.bank_id = r.bank_id
    GROUP BY b.bank_name
    ORDER BY avg_rating DESC
""")
print("\nAverage rating & sentiment per bank:")
for row in cur.fetchall():
    print(f"  {row[0]}: Rating={row[1]}, Sentiment={row[2]}")

# 3. Check for nulls in key columns
cur.execute("""
    SELECT 
        SUM(CASE WHEN review_text IS NULL THEN 1 ELSE 0 END) as null_reviews,
        SUM(CASE WHEN rating IS NULL THEN 1 ELSE 0 END) as null_ratings,
        SUM(CASE WHEN sentiment_label IS NULL THEN 1 ELSE 0 END) as null_sentiment
    FROM reviews
""")
row = cur.fetchone()
print(f"\nNull checks — reviews: {row[0]}, ratings: {row[1]}, sentiment: {row[2]}")

# 4. Theme distribution
cur.execute("""
    SELECT identified_theme, COUNT(*) as count
    FROM reviews
    GROUP BY identified_theme
    ORDER BY count DESC
""")
print("\nTheme distribution in database:")
for row in cur.fetchall():
    print(f"  {row[0]}: {row[1]:,}")

# 5. Total reviews
cur.execute("SELECT COUNT(*) FROM reviews")
total = cur.fetchone()[0]
print(f"\nTotal reviews in database: {total:,} ✅")

cur.close()
conn.close()

=== Database Verification ===

Reviews per bank:
  BOA: 600
  CBE: 600
  Dashen: 597

Average rating & sentiment per bank:
  CBE: Rating=4.10, Sentiment=0.283
  Dashen: Rating=3.93, Sentiment=0.299
  BOA: Rating=3.52, Sentiment=0.179

Null checks — reviews: 0, ratings: 0, sentiment: 0

Theme distribution in database:
  General Feedback: 1,246
  Transaction Performance: 202
  UI & App Design: 159
  Customer Support: 77
  Account Access Issues: 58
  Feature Requests: 55

Total reviews in database: 1,797 ✅


In [8]:
schema_content = """
-- bank_reviews database schema
-- Author: Bamla | 10 Academy x Kifiya | Week 2

CREATE TABLE IF NOT EXISTS banks (
    bank_id     SERIAL PRIMARY KEY,
    bank_name   VARCHAR(100) NOT NULL UNIQUE,
    app_name    VARCHAR(200),
    app_id      VARCHAR(200),
    avg_rating  DECIMAL(3,2),
    created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS reviews (
    review_id        INTEGER PRIMARY KEY,
    bank_id          INTEGER REFERENCES banks(bank_id),
    review_text      TEXT,
    rating           INTEGER CHECK (rating BETWEEN 1 AND 5),
    review_date      DATE,
    sentiment_label  VARCHAR(20),
    sentiment_score  DECIMAL(6,4),
    identified_theme VARCHAR(100),
    source           VARCHAR(50) DEFAULT 'Google Play',
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

os.makedirs('scripts', exist_ok=True)
with open('scripts/schema.sql', 'w') as f:
    f.write(schema_content)

print("✅ Schema saved to scripts/schema.sql")

✅ Schema saved to scripts/schema.sql
